In [1]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
import asyncio
from langchain_core.messages import HumanMessage, SystemMessage

from typing import TypedDict
from pydantic import BaseModel
import json

In [2]:

# Initialize BedrockChat
chat_model = ChatOllama(
    model="qwen2.5:32b",
    base_url="http://host.docker.internal:11434",
    verbose=True,
)



In [3]:
# Use the model
response = chat_model.invoke("Explain the concept of machine learning in simple terms.")
print(response.content)


Machine learning is like teaching a computer to learn from examples instead of being explicitly programmed with rules. Imagine you want to teach a child what a cat looks like. Instead of explaining all the features that define a cat (like fur, whiskers, and tail), you show them many pictures of cats. Over time, they start recognizing patterns and can identify a cat in new images.

Similarly, in machine learning, we feed data (like images or numbers) to an algorithm. The algorithm learns from this data by finding patterns and making predictions or decisions. With more data and practice, the computer gets better at its task—whether it's identifying cats in photos, predicting weather conditions, or recommending movies based on your past choices.

The key idea is that rather than telling a computer exactly what to do, we show it how to learn from examples so it can make decisions or predictions on its own.


In [4]:

# Use the model
response = await chat_model.ainvoke("capital of hawaii?")
print(response.content)


The capital of Hawaii is Honolulu. It is located on the island of Oahu and is not only the state capital but also the most populous city in the state of Hawaii.


In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://host.docker.internal:11434",
)

In [6]:
embeddings.embed_query("Hello, world!")

[0.015993318,
 -0.0006230358,
 -0.15882765,
 -0.0126873655,
 -0.017722998,
 0.061345775,
 -0.005486904,
 -0.010652253,
 -0.005597346,
 -0.040773835,
 0.013801117,
 0.07319954,
 0.019479591,
 0.0511815,
 0.02693512,
 -0.05958217,
 0.0075805094,
 -0.062439054,
 -0.02951461,
 0.025284873,
 -0.03170601,
 -0.08915614,
 0.0086086765,
 0.01985424,
 0.122950226,
 0.009043923,
 -0.037428264,
 0.07181813,
 0.012588832,
 -0.0032029068,
 -0.005140726,
 0.007733755,
 -0.0016500809,
 0.034073424,
 0.056595705,
 0.00039142702,
 0.022019839,
 0.0075600226,
 0.024588363,
 -0.026116889,
 0.012698393,
 -0.00064175733,
 0.012742669,
 0.005768861,
 0.075830124,
 -0.017261516,
 -0.019302353,
 -0.034168202,
 0.069966376,
 -0.035533763,
 -0.046175677,
 -0.0073280614,
 -0.0038491553,
 0.055153143,
 0.047077786,
 0.016622925,
 0.053463902,
 -0.035656884,
 0.018319312,
 0.055532426,
 0.041766953,
 0.05430201,
 0.028149717,
 0.032408964,
 0.0007777609,
 -0.04753623,
 -0.014109757,
 0.073530175,
 0.023691598,
 0.0

In [7]:
import numpy as np
np.array(embeddings.embed_documents(["Hello, world!", "Goodbye, world!"])).shape

(2, 768)

In [34]:
class TestReturn(BaseModel):
    capital: str
    location: str

# Initialize BedrockChat
chat_model2 = ChatOllama(
    model="qwen2.5:32b",
    base_url="http://host.docker.internal:11434",
    format=TestReturn.model_json_schema(),
    temperature=0,
)


structured_call = chat_model2.with_structured_output(TestReturn, method="json_schema")
response = structured_call.invoke(
    [
        HumanMessage("what is the capital of hawaii")
    ]
)
response



TestReturn(capital='Honolulu', location='Oahu island')

In [31]:
response.capital

'Honolulu'

In [12]:
TestReturn.model_json_schema()

{'properties': {'capital': {'title': 'Capital', 'type': 'string'},
  'location': {'title': 'Location', 'type': 'string'}},
 'required': ['capital', 'location'],
 'title': 'TestReturn',
 'type': 'object'}

In [13]:
def test_call2(this_model):

    response = this_model.invoke("what is capital of hawaii? only return the city's name")
    print(response)
    return response

my_result = test_call2(chat_model)
print(f"\nreturn from test_async_call:\n{my_result}")

content='Honolulu' additional_kwargs={} response_metadata={'model': 'qwen2.5:32b', 'created_at': '2025-08-23T21:45:32.613177Z', 'done': True, 'done_reason': 'stop', 'total_duration': 581501625, 'load_duration': 50735041, 'prompt_eval_count': 42, 'prompt_eval_duration': 431806625, 'eval_count': 3, 'eval_duration': 98208084, 'model_name': 'qwen2.5:32b'} id='run--8e1d20bd-34a6-4f71-9900-f499eb589910-0' usage_metadata={'input_tokens': 42, 'output_tokens': 3, 'total_tokens': 45}

return from test_async_call:
content='Honolulu' additional_kwargs={} response_metadata={'model': 'qwen2.5:32b', 'created_at': '2025-08-23T21:45:32.613177Z', 'done': True, 'done_reason': 'stop', 'total_duration': 581501625, 'load_duration': 50735041, 'prompt_eval_count': 42, 'prompt_eval_duration': 431806625, 'eval_count': 3, 'eval_duration': 98208084, 'model_name': 'qwen2.5:32b'} id='run--8e1d20bd-34a6-4f71-9900-f499eb589910-0' usage_metadata={'input_tokens': 42, 'output_tokens': 3, 'total_tokens': 45}


In [14]:
from langchain_openai import ChatOpenAI

openai_model = ChatOpenAI(
    model="gpt-4o-mini"
)

In [17]:
class TestReturn(BaseModel):
        response_text: str

structured_llmodel = openai_model.with_structured_output(TestReturn)
response = structured_llmodel.invoke("what is capital of hawaii? only return the city's name.")
print(response)

response_text='Honolulu'


In [ ]:
print(structured_llmodel.invoke("hello"))



In [21]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel

# Define your output schema
class Person(BaseModel):
    name: str
    age: int

# Instantiate the Ollama LLM (replace with 'gpt-oss' or your model)
llm = ChatOllama(
    model="qwen2.5:32b",  # or your local gpt-oss install
    base_url="http://host.docker.internal:11434",
    temperature=.5,
    format="json"
)# .with_structured_output(Person, method="json_schema", include_raw=True)
#
response = llm.invoke("the person is Alice is 31 years-old. what is the person name and age.")
# response = llm.invoke("what is the capital of hawaii?")
print(response.content)


{
  "name": "Alice",
  "age": 31
}


In [28]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel

# Define your desired output schema
class Add(BaseModel):
    a: int
    b: int

# Instantiate ChatOllama
llm = ChatOllama(
    model="qwen2.5:32b",
    base_url="http://host.docker.internal:11434",
)

# Use with_structured_output with the json_schema method
structured_llm = llm.with_structured_output(Add, method="json_schema",)

print(f"return the values of 'a' and 'b': a=1, b=2. return as json with schema {Add.model_json_schema()}")

# Invoke the model
structured_llm.invoke(f"return the values of 'a' and 'b': a=1, b=2. return as json with schema {Add.model_json_schema()}")


return the values of 'a' and 'b': a=1, b=2. return as json with schema {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'Add', 'type': 'object'}


Add(a=1, b=2)